In [ ]:
import arcpy
from arcpy.sa import ExtractByMask

# Check out Spatial Analyst
arcpy.CheckOutExtension("Spatial")

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
GBC_GDB  = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec.gdb"
USC_GDB  = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\USCaribbean_Jeremiah_10meter.gdb"

# Input rasters (read-only source GDBs – never written to)
GBC_bathy = GBC_GDB + r"\GBC15a_bathy"
GBC_slope = GBC_GDB + r"\GBC15a_slope"
USC_bathy = USC_GDB + r"\USC10m_bathy"
USC_slope = USC_GDB + r"\USC10m_slope"

# Mask polygon (must already exist in your project or a GDB you own)
boundary = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\arcgis_UVI2_Around_STX_20250306\arcgis_UVI2_Around_STX_20250306.gdb\aroundstx2025_boundary"

# Output GDB – write all results here, NOT to the source GDBs
OUT_GDB  = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Files\THESIS_FILES\Missions\arcgis_UVI2_Around_STX_20250306\arcgis_UVI2_Around_STX_20250306.gdb"

# ── Environment ──────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput    = True
arcpy.env.compression        = "NONE"
arcpy.env.workspace          = OUT_GDB

In [35]:
# ── Helper: Run Extract by Mask and save result to OUT_GDB ───────────────────
def extract(in_raster, mask, out_name, snap_raster, label=""):
    arcpy.env.snapRaster = snap_raster
    print(f"  Extracting: {label} → {out_name}")
    result = ExtractByMask(in_raster, mask, "INSIDE")
    result.save(OUT_GDB + "\\" + out_name)
    print(f"  Saved: {OUT_GDB}\\{out_name}")

In [ ]:
# ── 1A. GEBCO – base bathymetry ────────────────────────────────────────────────
print("=== GEBCO: base bathymetry ===")
extract(
    in_raster   = GBC_bathy,
    mask        = boundary,
    out_name    = "GBC15a_bathy_aroundstx2025",
    # snap to original before first cut
    snap_raster = GBC_bathy,
    label       = "GBC15a_bathy → aroundstx2025"
)

In [ ]:
# ── 1B. GEBCO – shaded relief bathymetry (duplicate crop) ──────────────────────
print("=== GEBCO: shaded relief bathymetry ===")
extract(
    in_raster   = GBC_bathy,
    mask        = boundary,
    out_name    = "GBC15a_SRbathy_aroundstx2025",
    snap_raster = GBC_bathy,
    label       = "GBC15a_bathy → aroundstx2025 for SR"
)

In [ ]:
# ── 2. GEBCO – slope ──────────────────────────────────────────────────────────
print("=== GEBCO: slope ===")
extract(
    in_raster   = GBC_slope,
    mask        = boundary,
    out_name    = "GBC15a_slope_aroundstx2025",
    snap_raster = GBC_slope,
    label       = "GBC15a_slope → aroundstx2025"
)

In [ ]:
# ── 3A. USC – base bathymetry ──────────────────────────────────────────────────
print("=== USC 10 m: base bathymetry ===")
extract(
    in_raster   = USC_bathy,
    mask        = boundary,
    out_name    = "USC10m_bathy_aroundstx2025",
    snap_raster = USC_bathy,
    label       = "USC10m_bathy → aroundstx2025"
)

In [ ]:
# ── 3B. USC – shaded relief bathymetry (duplicate crop) ────────────────────────
print("=== USC 10 m: shaded relief bathymetry ===")
extract(
    in_raster   = USC_bathy,
    mask        = boundary,
    out_name    = "USC10m_SRbathy_aroundstx2025",
    snap_raster = USC_bathy,
    label       = "USC10m_bathy → aroundstx2025 for SR"
)

In [ ]:
# ── 4. USC – slope ────────────────────────────────────────────────────────────
print("=== USC: slope ===")
extract(
    in_raster   = USC_slope,
    mask        = boundary,
    out_name    = "USC10m_slope_aroundstx2025",
    snap_raster = USC_slope,
    label       = "USC10m_slope → aroundstx2025"
)

In [ ]:
# ── 5. Add all layers to the current map ─────────────────────────────────────
print("=== Adding layers to map ===")

aprx = arcpy.mp.ArcGISProject("CURRENT")
m    = aprx.activeMap

# Extracted outputs
for out_name in [
    "GBC15a_bathy_aroundstx2025",
    "GBC15a_SRbathy_aroundstx2025",
    "GBC15a_slope_aroundstx2025",
    "USC10m_bathy_aroundstx2025",
    "USC10m_SRbathy_aroundstx2025",
    "USC10m_slope_aroundstx2025"
]:
    m.addDataFromPath(OUT_GDB + "\\" + out_name)
    print(f"  Added output: {out_name}")

arcpy.CheckInExtension("Spatial")
print("=== Done ===")